# Quick Start: From CSV to Report in 5 Minutes

This tutorial shows you the fastest path to results. If you have your bills in a CSV file, you can go from data to a complete report in just a few lines of code.

**What you'll learn:**
- How to format your CSV file
- Loading bills from CSV
- Generating a report with defaults
- Interpreting the results

This is perfect if you want to get started quickly without diving into all the details.


## Step 1: Prepare Your CSV File

Your CSV file needs these columns:

**Required columns:**
- `bill_id`: Unique identifier (e.g., "prop_tax")
- `service`: Descriptive name (e.g., "Property Tax")
- `amount_due`: Amount as a number (e.g., 3600.00)
- `recurring`: True or False

**For one-time bills, also include:**
- `due_date`: Date in YYYY-MM-DD format (e.g., 2025-11-01)

**For recurring bills, also include:**
- `start_date`: First occurrence date (e.g., 2025-04-24)
- `frequency`: One of "daily", "weekly", "monthly", "quarterly", "annual"
- `interval`: Number (optional, defaults to 1)

Let's look at an example CSV file:


In [ ]:
# Here's an example of what your CSV data structure should look like.
# You can also use Python dictionaries (shown below) or other file formats.

sample_bill_data = [
    {
        'bill_id': 'prop_tax',
        'service': 'Property Tax',
        'amount_due': 3600.00,
        'recurring': True,
        'start_date': '2025-11-01',  # YYYY-MM-DD format
        'frequency': 'annual',
        'interval': 1
    },
    {
        'bill_id': 'car_ins',
        'service': 'Car Insurance',
        'amount_due': 750.00,
        'recurring': True,
        'start_date': '2025-04-24',
        'frequency': 'monthly',
        'interval': 6
    }
]

print("Sample bill data structure:")
for bill in sample_bill_data:
    print(f"  {bill}")


## Step 2: Import and Create Your Fund

Just two imports needed:


In [ ]:
from datetime import date
from sinkingfund import SinkingFund


## Step 3: Create Fund and Load Bills

Create your sinking fund with a planning period and initial balance, then load bills from your CSV file:


In [ ]:
# Create sinking fund for 2025.
fund = SinkingFund(
    start_date=date(2025, 1, 1),
    end_date=date(2025, 12, 31),
    balance=5000.00  # Your starting balance
)

# Load bills from dictionary data (or use CSV/Excel/JSON file path).
sample_bills = [
    {
        'bill_id': 'prop_tax',
        'service': 'Property Tax',
        'amount_due': 3600.00,
        'recurring': True,
        'start_date': date(2025, 11, 1),
        'frequency': 'annual',
        'interval': 1
    },
    {
        'bill_id': 'car_ins',
        'service': 'Car Insurance',
        'amount_due': 750.00,
        'recurring': True,
        'start_date': date(2025, 4, 24),
        'frequency': 'monthly',
        'interval': 6
    }
]

fund.create_bills(sample_bills)

print(f"Loaded {len(fund.bill_manager.bills)} bills")
print(f"Planning period: {fund.start_date} to {fund.end_date}")
print(f"Initial balance: ${fund.balance}")


## Step 4: Generate Your Report

One method call does everything - creates envelopes, allocates funds, sets up schedules, and generates the report:


In [ ]:
# Generate complete report with bi-weekly contributions.
report = fund.quick_report(contribution_interval=14)

print("Report generated!")
print(f"  Covers {len(report)} days")
print(f"  From {min(report.keys())} to {max(report.keys())}")


## Step 5: Explore Your Report

The report is a dictionary where each date contains account balance, contributions, and payouts. Let's look at key information:


In [ ]:
# Summary statistics.
total_contributions = sum(
    day['contributions']['total'] 
    for day in report.values()
)
total_payouts = sum(
    day['payouts']['total'] 
    for day in report.values()
)
contribution_days = [
    date for date, data in report.items() 
    if data['contributions']['total'] > 0
]

print("=== Report Summary ===")
print(f"Total contributions: ${total_contributions:.2f}")
print(f"Total payouts: ${total_payouts:.2f}")
print(f"Number of contribution days: {len(contribution_days)}")
print(f"Average contribution: ${total_contributions / len(contribution_days):.2f}")


Let's look at a few specific days to understand the data:


In [ ]:
# First day of planning period.
first_day = report[date(2025, 1, 1)]
print("=== January 1, 2025 (Start) ===")
print(f"Account Balance: ${first_day['account_balance']['total']:.2f}")
print(f"Contributions: ${first_day['contributions']['total']:.2f}")
print(f"Payouts: ${first_day['payouts']['total']:.2f}")

# First contribution day.
first_contrib = contribution_days[0]
contrib_day = report[first_contrib]
print(f"\n=== {first_contrib} (First Contribution) ===")
print(f"Account Balance: ${contrib_day['account_balance']['total']:.2f}")
print(f"Contributions: ${contrib_day['contributions']['total']:.2f}")
if contrib_day['contributions']['total'] > 0:
    print("Contribution breakdown:")
    for bill_id, amount in contrib_day['contributions']['bills'].items():
        if amount > 0:
            print(f"  {bill_id}: ${amount:.2f}")
